# Widgets and an editable table

Widgets in the `morris_lisp` kernel: sliders and buttons that call Lisp, and a grid to edit a table in. Needs `ipywidgets` and `ipydatagrid`. The manual's "Widgets in a notebook" has all of it.

## A mortgage payment, as sliders move

A function that a widget calls has no cell to write to, so `:output out` says where what it displays goes.

In [ ]:
(define out (widget-output))

(define (show-payment)
  (widget-clear out)
  (widget-capture out
    (lambda ()
      (display (format "Monthly payment: {:,.2f}\n"
                       (payment (/ (widget-get rate "value") 1200)
                                (* 12 (widget-get years "value"))
                                (widget-get balance "value")))))))

(define balance (widget-slider 50000 1000000 :value 300000 :step 10000 :description "balance" :on-change (lambda (v) (show-payment))))
(define rate    (widget-slider 1 10 :value 6.5 :step 0.125 :description "rate %" :on-change (lambda (v) (show-payment))))
(define years   (widget-slider 5 40 :value 30 :description "years" :on-change (lambda (v) (show-payment))))
(show-payment)
(widget-column balance rate years out)

## Editing a table

`table-grid` makes a grid of a table. Double-click a cell to change it; click a heading to sort. `grid-on-edit` hears each change, and `grid-table` gives the table as it is now.

In [ ]:
(define loans
  (make-table "id"      (vector "A-1" "A-2" "A-3")
              "balance" (vector 250000.0 410000.0 175500.0)
              "rate"    (vector 0.055 0.0625 0.0475)
              "state"   (vector "NY" "CA" "TX")))

(define grid (table-grid loans))
(define log (widget-output))
(grid-on-edit grid
              (lambda (column row value) (display (format "{} of loan {} is now {}\n" column row value)))
              :output log)
(widget-column grid log)

Change some cells, then press the button: it shows the edited table, and the total balance of it.

In [ ]:
(define report (widget-output))

(define (show-edits)
  (widget-clear report)
  (widget-capture report
    (lambda ()
      (let ((now (grid-table grid)))
        (display-table now '(("balance" ",.2f") ("rate" ".3%")))
        (display (format "total balance {:,.2f}\n" (vector-sum (table-column now "balance"))))))))

(widget-column (widget-button "Show the table" (lambda () (show-edits)) :button-style "primary") report)